# Fluss Python Client Demo: From Source to Server

This notebook will guide you through the **exact workflow** of how to test your custom Rust changes to the Fluss Python client against a real running Fluss server.

In a real-world scenario, you would normally just `pip install fluss`. But because you are *developing* the library itself (for Issue #424), you must compile your own version from source, start the Fluss server in the background, and then connect to it.

---

In [ ]:
import sys

In [ ]:
# 1. Install maturin specifically into the Notebook's Python environment
!{sys.executable} -m pip install maturin

In [ ]:
# 2. Compile and install your local fluss-rust code into this environment
!cd /Users/jaredyu/Desktop/open_source/fluss-rust/bindings/python && {sys.executable} -m maturin develop

## Step 1: Compile the Python Client from Source

Before we even touch Python or start a server, we need to compile the Rust code into a Python module. We use `maturin` for this.

**Objective:** Build the `fluss-rust` project (`bindings/python`) and inject it into your current environment.

*Note: Run this in your terminal, or use the `!` in Jupyter to execute it in the shell.*

In [ ]:
# !cd /Users/jaredyu/Desktop/open_source/fluss-rust/bindings/python && maturin develop

If that succeeds, you now have a local, custom version of the `fluss` Python package installed that includes whatever Rust changes you just made!

---

## Step 2: Start the Fluss Server (The Background Process)

Fluss is a distributed system, much like Kafka or databases. The Python client we just built is only the messenger; it needs a server to talk to.

**Objective:** Start a local, standalone Fluss server cluster.

You will need the actual **Fluss server release** (which is written in Java and uses ZooKeeper). If you don't have it downloaded, you generally download it from the Apache website or build it from the main `fluss` repo (not `fluss-rust`).

Assuming you have downloaded and extracted it to `/path/to/fluss-server`:

In [ ]:
# # This is a terminal command. DO NOT run this directly in the notebook unless you want it to hang!
# # Open a separate terminal window and run:

# !echo "Run this in a regular terminal: /path/to/fluss-server/bin/start-cluster.sh"

Once started, the server usually runs on `localhost:9092` or `localhost:8081` (check the output logs in your terminal).

Leave that terminal open. The server is now running in the background!

In [ ]:
# import os

# FLUSS_VERSION = "0.9.0"
# FLUSS_DIR = f"/tmp/fluss-{FLUSS_VERSION}-incubating"
# DOWNLOAD_URL = f"https://downloads.apache.org/incubator/fluss/fluss-{FLUSS_VERSION}-incubating/fluss-{FLUSS_VERSION}-incubating-bin.tgz"

# # if not os.path.exists(FLUSS_DIR):
# #     print(f"Downloading Apache Fluss {FLUSS_VERSION}...")
# #     !wget -qO /tmp/fluss.tgz {DOWNLOAD_URL}
# #     print("Extracting...")
# #     !tar -xzf /tmp/fluss.tgz -C /tmp
# #     !rm /tmp/fluss.tgz
# #     print(f"Fluss server extracted to {FLUSS_DIR}")
# # else:
# #     print(f"Fluss server already exists at {FLUSS_DIR}")

Once extracted, start the standalone server. DO NOT run this directly in the notebook unless you want it to block! Open a separate terminal window and run:

cd /tmp/fluss-0.9.0-incubating && ./bin/start-cluster.sh

---

## Step 3: Connect and Interact via Python

Now the magic happens. We are going to import the library we compiled in Step 1, connect to the server we started in Step 2, and test our code.

This is where you will eventually test your `async for` implementation for Issue #424!

In [ ]:
# import asyncio

# # This imports the module you compiled with `maturin develop`!
# from fluss import FlussConnection, Config, TablePath

# async def run_demo():
#     # 1. Connect to the local server running in the background
#     print("Connecting to local Fluss cluster...")
    
#     # Configure the client to connect to localhost
#     conf = Config()
#     # conf.bootstrap_servers = "localhost:8081" # or 9092, depending on server logs
#     conf.bootstrap_servers = "localhost:9123"
    
#     # Create the connection
#     client = await FlussConnection.create(conf) 
#     print("Connected!")

#     # 2. Interact with the server (e.g., getting a scanner for a table)
#     # The default database is typically "fluss" or "default"
#     table_path = TablePath("fluss", "my_test_table") 
    
#     try:
#         admin = await client.get_admin()
        
#         # Make sure the table exists before grabbing it
#         if not await admin.table_exists(table_path):
#             print(f"Table {table_path} does not exist yet! Go create one.")
#             return

#         table = await client.get_table(table_path)
#         scanner = await table.new_scan().create_log_scanner()
        
#     except Exception as e:
#         print(f"Error setting up scanner: {e}")
#         return

#     # 3. THIS IS THE TESTING GROUND FOR ISSUE #424!
#     # If you successfully implemented `__aiter__` and `__anext__` in Rust,
#     # this Python syntax will work flawlessly.
#     print("Starting async scan...")
    
#     # async for record in scanner:
#     #     print(f"Received record: {record}")
        
#     print("Demo complete!")

In [ ]:
# Execute the async run loop directly in the Jupyter Notebook
# await run_demo()

In [ ]:
import asyncio
from fluss import FlussConnection, Config, TablePath, TableDescriptor, Schema, EARLIEST_OFFSET
import pyarrow as pa

async def run_demo():
    print("Connecting to local Fluss cluster...")
    conf = Config()
    conf.bootstrap_servers = "localhost:9123" # The port we discovered
    client = await FlussConnection.create(conf) 
    print("Connected!")

    table_path = TablePath("fluss", "my_test_table")
    admin = await client.get_admin()
    
    # --- 1. SET UP THE TABLE ---
    if not await admin.table_exists(table_path):
        print(f"Creating table {table_path}...")
        
        # Define the schema using PyArrow
        pa_schema = pa.schema([
            pa.field("id", pa.int32()),
            pa.field("name", pa.string()),
        ])
        
        # Create Fluss schema (no primary key for a simple log table)
        schema = Schema(pa_schema)
        
        # Create table descriptor
        descriptor = TableDescriptor(schema)
        
        # Create the table
        await admin.create_table(table_path, descriptor)
        print("Table created successfully!")
        
        # Insert some mock data so the scanner has something to read!
        print("Inserting mock data...")
        table = await client.get_table(table_path)
        writer = table.new_append().create_writer()
        
        # Append rows manually
        writer.append({"id": 1, "name": "Alice"})
        writer.append({"id": 2, "name": "Bob"})
        writer.append({"id": 3, "name": "Charlie"})
        
        # Flush to ensure data is written to the server
        await writer.flush()
        print("Mock data inserted!")
    else:
        print(f"Table {table_path} already exists. Skipping creation.")

    # --- 2. THE TESTING GROUND FOR ISSUE #424 ---
    print("\nSetting up scanner...")
    table = await client.get_table(table_path)
    scanner = await table.new_scan().create_log_scanner()
    
    # We must explicitly subscribe the scanner to a bucket and starting offset
    print("Subscribing to bucket 0 at EARLIEST_OFFSET")
    scanner.subscribe(0, EARLIEST_OFFSET)

    print("Starting async scan...\n")
    
    # If you successfully implemented `__aiter__` and `__anext__` in Rust,
    # this Python syntax will work flawlessly.
    
    # async for record in scanner:
    #     print(f"Received record: {record}")
        
    print("Demo complete!")

In [ ]:
# Execute the async run loop directly in the Jupyter Notebook
await run_demo()

## The Iteration Cycle

If your `async for` loop throws an error (e.g., `TypeError: 'LogScanner' object is not an async iterable`):

1. Go back to your Rust IDE (`bindings/python/src/table.rs`).
2. Fix the `__aiter__` or `__anext__` logic.
3. Run **Step 1** (`maturin develop`) again to recompile.
4. Restart the Jupyter Kernel (to load the fresh binary).
5. Run **Step 3** again.

*(You do NOT need to restart the Fluss server in Step 2. It can happily stay running in the background!)*